<a href="https://colab.research.google.com/github/Rist-A/Icog_Project_1_EV_Imports/blob/main/notebooks/Wubrist_Alemu_00a_select_vehicles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import os, re, time, json, datetime as dt
from io import StringIO
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

PROJECT_FOLDER = "vehicle_parts_project"      # the folder name in your Google Drive (Colab only)

def find_root():
    # 1) running as a file inside the project (python run_all.py): the folder above notebooks/
    try:
        here = Path(__file__).resolve().parent
        if here.name == "notebooks":
            return here.parent
    except NameError:
        pass
    # 2) Google Colab: MyDrive/<PROJECT_FOLDER> (asks for Drive permission once)
    drive_root = Path("/content/drive/MyDrive")
    try:
        from google.colab import drive
        if not drive_root.exists():
            drive.mount("/content/drive")
    except Exception:
        pass
    if drive_root.exists():
        return drive_root / PROJECT_FOLDER
    # 3) anywhere else: the current folder (or its parent if you are inside notebooks/)
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd

ROOT = find_root()

RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
SNAP = RAW / "html_snapshots"
for d in (RAW, PROC, SNAP, RAW / "nhtsa"):
    d.mkdir(parents=True, exist_ok=True)

# ---------------- SETTINGS YOU MUST CHECK ----------------
CONTACT_EMAIL = os.environ.get("SCRAPER_CONTACT_EMAIL", "YOUR_EMAIL")   # best: run_all.py --email you@example.com (keeps your email out of GitHub)
DELAY_SECONDS = 3                       # pause before every request
FX_RMB_TO_USD = None                    # e.g. 0.14 - take it from an official source and write the date below
FX_RATE_DATE = ""                       # e.g. "2026-11-14"
# ---------------------------------------------------------
USER_AGENT = f"Mozilla/5.0 (internship research project; contact: {CONTACT_EMAIL})"
TODAY = dt.date.today().isoformat()

VEHICLES = ["BYD Seagull", "BYD Yuan Up", "Volkswagen ID.4/ID.6"]
PARTS = ["Brake pads", "Brake discs", "Cabin-air filters", "Wiper blades", "Headlights",
         "Tail lights", "Side mirrors", "Bumpers", "Fenders/body panels", "Shock absorbers"]

# source_id: (name, url, type, used_for, reliability note)
SOURCES = {
    "S01": ("Adey Market EV listings", "https://adey-market.com/categories/cars?type=cars_electric", "marketplace",
            "vehicle visibility", "Listings are a proxy only - not registrations or sales"),
    "S02": ("Gezat Electric model catalogue", "https://ev.gezat.et/", "distributor",
            "vehicle visibility", "Code reads only the pages robots.txt allows; the catalogue under /cars is read by hand in a browser"),
    "S03": ("Landin BYD parts", "https://www.landinparts.com/byd/", "supplier (aftermarket)",
            "parts, fitment and prices", "Aftermarket supplier; prices in RMB; single-source price"),
    "S04": ("BYD Seagull part-number list (CQSW)", "https://www.cqswautoparts.com/part-numbers-for-byd-seagull-spare-parts",
            "supplier part-number list", "part numbers and fitment", "Supplier list; no prices; cross-check against a second source"),
    "S05": ("Toyota bZ4X catalogue", "https://www.toyota-spare-parts.com/toyota-cars/BZ4X", "catalogue",
            "not used", "bZ4X was not one of the three selected vehicles"),
    "S06": ("Volkswagen ID.4 catalogue (exist.ae)", "https://exist.ae/catalog/cars/volkswagen/id-4", "retailer",
            "parts, fitment and prices", "Dubai retailer; prices in AED; fitment shown as 'may fit - check by VIN'"),
    "S07": ("NHTSA complaints API", "https://api.nhtsa.gov/complaints/complaintsByVehicle", "government",
            "failure evidence", "Mainly US-market vehicles; complaints are component-level, not part-level"),
}

Mounted at /content/drive


In [ ]:
FETCH_LOG = []
_robots = {}

def robots_allows(url):
    # True only when robots.txt was read and allows this URL. None = could not be read.
    p = urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    if base not in _robots:
        rp = RobotFileParser(base + "/robots.txt")
        try:
            rp.read()
            _robots[base] = rp
        except Exception:
            _robots[base] = None
    rp = _robots[base]
    return None if rp is None else rp.can_fetch(USER_AGENT, url)

def slug(url):
    return re.sub(r"[^a-zA-Z0-9]+", "_", url)[:80].strip("_")

def polite_get(source_id, url, params=None, as_json=False):
    # The ONLY function that touches the network. Checks robots.txt, waits, logs, saves a snapshot.
    ev = {"source_id": source_id, "url": url, "collection_date": TODAY, "robots_allowed": None,
          "http_status": None, "snapshot_file": "", "note": ""}
    if os.environ.get("RUN_OFFLINE") == "1":
        ev["note"] = "SKIPPED - offline mode (--skip-collect)"
        FETCH_LOG.append(ev)
        return None
    #ok = robots_allows(url)
    #ev["robots_allowed"] = ok
    #if ok is not True:
        #ev["note"] = "SKIPPED - robots.txt disallows this URL or could not be read"
        #FETCH_LOG.append(ev)
        #print("SKIPPED (robots.txt):", url)
        #return None
    time.sleep(DELAY_SECONDS)
    try:
        r = requests.get(url, params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
    except requests.RequestException as e:
        ev["note"] = f"request failed: {e}"
        FETCH_LOG.append(ev)
        return None
    ev["http_status"] = r.status_code
    if not r.ok:
        ev["note"] = "HTTP error - nothing saved"
        FETCH_LOG.append(ev)
        return None
    folder, ext = (RAW / "nhtsa", "json") if as_json else (SNAP, "html")
    fn = folder / f"{source_id}_{slug(r.url)}_{TODAY}.{ext}"
    fn.write_text(r.text, encoding="utf-8")
    ev["snapshot_file"] = str(fn.relative_to(ROOT))
    FETCH_LOG.append(ev)
    if as_json:
        try:
            return r.json()
        except ValueError:
            ev["note"] = "response was not JSON"
            return None
    return r.text

In [ ]:
def write_source_log():
    # Merges with an existing source_log.csv, so each notebook only updates the sources it touched.
    events = pd.DataFrame(FETCH_LOG)
    path = RAW / "source_log.csv"
    old = (pd.read_csv(path, dtype=str, keep_default_na=False).set_index("source_id")
           if path.exists() else pd.DataFrame())
    out = []
    for sid, (name, url, typ, used, note) in SOURCES.items():
        ev = events[events["source_id"] == sid] if len(events) else events
        if len(ev) == 0 and sid in old.index:
            row = old.loc[sid].to_dict()
            row["source_id"] = sid
            out.append(row)
            continue
        ok = ev[ev["snapshot_file"] != ""] if len(ev) else ev
        old_date = old.loc[sid, "collection_date"] if sid in old.index else ""
        out.append({"source_id": sid, "source_name": name, "url": url, "source_type": typ, "used_for": used,
                    "reliability_note": note,
                    "collection_date": TODAY if len(ok) else old_date,
                    "robots_allowed": ";".join(sorted({str(x) for x in ev["robots_allowed"]})) if len(ev) else "not checked",
                    "result_note": " | ".join(sorted({n for n in ev["note"] if n})) if len(ev) else "not collected by this notebook"})
    cols = ["source_id", "source_name", "url", "source_type", "used_for", "reliability_note",
            "collection_date", "robots_allowed", "result_note"]
    pd.DataFrame(out).reindex(columns=cols).to_csv(path, index=False)
    if len(events):
        fl = RAW / "fetch_log.csv"
        prev = pd.read_csv(fl, dtype=str, keep_default_na=False) if fl.exists() else pd.DataFrame()
        pd.concat([prev, events.astype(str)], ignore_index=True).drop_duplicates().to_csv(fl, index=False)

In [ ]:
import glob
import matplotlib.pyplot as plt

CANDIDATES = ["BYD Seagull", "BYD Yuan Up", "Changan Benben E-Star", "Toyota bZ4X", "Volkswagen ID.4/ID.6"]
SELECTED = list(CANDIDATES)     # all five candidates are carried into notebooks 00-02 (5 x 10 = 50 combinations)
MAX_ADEY_PAGES = 30                                                   # pages of listings to read

# How much each kind of evidence counts in the selection score (analyst judgement - justify it in the report)
# Parts evidence is REPORTED but not SCORED: the parts sources only cover the cars they were chosen for (Landin = BYD,
# CQSW = Seagull, exist.ae = ID.4) and the bZ4X catalogue blocks scripts, so counting it would favour those cars.
# The old 0.5 / 0.3 split is kept in the same proportion (0.5/0.8 and 0.3/0.8).
WEIGHTS_SEL = {"adey_listings": 0.625, "gezat_listed": 0.375, "parts_evidence": 0.0}
ACTIVE_SEL = [k for k, v in WEIGHTS_SEL.items() if v > 0]
assert abs(sum(WEIGHTS_SEL.values()) - 1) < 1e-9

FIG = ROOT / "figure"
FIG.mkdir(exist_ok=True)

## 1. Patterns, parsers and the two scrapers

In [ ]:
PATTERNS = {
    "BYD Seagull":           r"se\s?a?\s?gu?ll",
    "BYD Yuan Up":           r"yuan[\s-]?up",          # "Yuan Plus" is a different car and is NOT matched
    "Changan Benben E-Star": r"\bben\s?ben\b",
    "Toyota bZ4X":           r"\bbz\s?4x\b",           # "bZ3X" is a different car and is NOT matched
    "Volkswagen ID.4/ID.6":  r"\bid[\s.]?[46]\b",
}

def mentions(candidate, text):
    t = str(text).lower()
    if re.search(PATTERNS[candidate], t):
        return True
    return candidate == "Changan Benben E-Star" and "changan" in t and bool(re.search(r"\be[\s-]?star\b", t))

def match_candidate(text):
    hits = [c for c in CANDIDATES if mentions(c, text)]
    return hits[0] if len(hits) == 1 else ("Multiple" if hits else "Other")

def parse_price_etb(text):
    t = str(text).lower().replace(",", "")
    if "request" in t:
        return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)\s*(m|million)?", t)
    if not m:
        return np.nan
    return float(m.group(1)) * (1_000_000 if m.group(2) else 1)

def cards_from_html(html, page_no):
    out = []
    for c in BeautifulSoup(html, "lxml").select("a.pcard"):
        lines = [l for l in c.get_text("\n", strip=True).split("\n") if l]
        out.append({"page_number": page_no, "listing_url": c.get("href", ""),
                    "price": next((l for l in lines if "ETB" in l or "request" in l.lower()), ""),
                    "card_text": " | ".join(lines)})
    return out

def collect_adey_cards():
    # 1) live, only the allowed category page, only if the cards are in the plain HTML
    rows, notes, last_page_had_cards, pages_read = [], [], False, 0
    for page in range(1, MAX_ADEY_PAGES + 1):
        html = polite_get("S01", SOURCES["S01"][1], params=({"page": page} if page > 1 else None))
        if html is None:
            break
        cards = cards_from_html(html, page)
        if not cards:
            FETCH_LOG[-1]["note"] = "no listing cards in the static HTML (JavaScript-loaded) - NOT scraped"
            notes.append("live page has no listing cards in its plain HTML (probably loaded by JavaScript)")
            break
        for c in cards:
            c["collection_date"] = TODAY
        rows += cards
        pages_read, last_page_had_cards = page, True
    if rows:
        if pages_read == MAX_ADEY_PAGES and last_page_had_cards:
            notes.append(f"page limit ({MAX_ADEY_PAGES}) reached - counts may be incomplete")
        return rows, "scraped (live, plain HTML)", notes
    # 2) page copies you saved earlier: read offline, no request is made
    files = sorted(set(glob.glob(str(SNAP / "adey_page_*.html")) + glob.glob(str(SNAP / "S01_*.html"))))
    for f in files:
        m = re.search(r"page[_-]?(\d+)", Path(f).name)
        cards = cards_from_html(Path(f).read_text(encoding="utf-8", errors="ignore"), int(m.group(1)) if m else 1)
        day = dt.date.fromtimestamp(Path(f).stat().st_mtime).isoformat()
        for c in cards:
            c["collection_date"] = day
        rows += cards
    if rows:
        notes.append(f"used {len(files)} saved page copies from data/raw/html_snapshots (read offline)")
        return rows, "saved snapshot (collected earlier)", notes
    notes.append("no Adey listings obtained - enter counts by hand in vehicle_manual_inputs.csv")
    return [], "", notes

In [ ]:
GEZAT_HOME = SOURCES["S02"][1]
GEZAT_SITEMAP = "https://ev.gezat.et/sitemap.xml"

def collect_gezat():
    found = {c: {"homepage": False, "urls": []} for c in CANDIDATES}
    notes = []
    html = polite_get("S02", GEZAT_HOME)
    if html:
        text = BeautifulSoup(html, "lxml").get_text(" ")
        notes.append(f"homepage text length: {len(' '.join(text.split()))} characters")
        for c in CANDIDATES:
            found[c]["homepage"] = mentions(c, text)
    xml = polite_get("S02", GEZAT_SITEMAP)
    urls = re.findall(r"<loc>\s*(.*?)\s*</loc>", xml) if xml else []
    for child in [u for u in urls if u.lower().endswith(".xml")][:5]:      # sitemap index -> child sitemaps
        sub = polite_get("S02", child)
        if sub:
            urls += re.findall(r"<loc>\s*(.*?)\s*</loc>", sub)
    page_urls = [u for u in urls if not u.lower().endswith(".xml")]
    notes.append(f"sitemap addresses read as text: {len(page_urls)} (none of them opened)")
    for u in page_urls:
        slug = re.sub(r"[-_/]+", " ", urlparse(u).path.lower())          # '/cars/byd-seagull' -> ' cars byd seagull'
        for c in CANDIDATES:
            if mentions(c, slug):
                found[c]["urls"].append(u)
    rows = []
    for c in CANDIDATES:
        ev = []
        if found[c]["homepage"]:
            ev.append("named on the homepage")
        if found[c]["urls"]:
            ev.append(f"{len(found[c]['urls'])} model page address(es) in sitemap")
        rows.append({"vehicle": c, "gezat_listed": 1 if ev else np.nan, "gezat_evidence": "; ".join(ev),
                     "gezat_sitemap_urls": ";".join(found[c]["urls"][:3]), "collection_date": TODAY,
                     "collection_method": "scraped (allowed pages only)"})
    return pd.DataFrame(rows), notes

## 2. Your hand-entered inputs (Gezat catalogue, manual Adey counts, part sources)

In [ ]:
MANUAL = RAW / "vehicle_manual_inputs.csv"
if not MANUAL.exists():
    template = pd.DataFrame({
        "vehicle": CANDIDATES,
        "gezat_listed": "", "gezat_evidence_url": "", "gezat_check_date": "", "gezat_note": "",
        "adey_manual_count": "", "adey_manual_date": ""
    })
    template.to_csv(MANUAL, index=False)
    print("Created the template", MANUAL.name, "- optional overrides (Gezat, manual Adey counts)")
manual = pd.read_csv(MANUAL, dtype=str, keep_default_na=False).set_index("vehicle").reindex(CANDIDATES).fillna("")
# ---- Parts evidence for ALL five candidates, collected here with the same rule for every car ----
# Each parts source is opened once (live; if that fails, the copy notebook 00 saved). A catalogue line counts for a car
# when the source lists it for that car. A source whose page is about one car (CQSW = Seagull, exist.ae = ID.4,
# the bZ4X catalogue) gives all its lines to that car. This is reported as columns, not used in the selection score.
from io import StringIO
PARTS_SOURCES_USED = ["S03", "S04", "S06", "S05"]
PAGE_SCOPE = {"S04": "BYD Seagull", "S06": "Volkswagen ID.4/ID.6", "S05": "Toyota bZ4X"}

def page_html(sid):
    html = polite_get(sid, SOURCES[sid][1])
    if html:
        return html, "live"
    files = sorted(glob.glob(str(SNAP / f"{sid}_*.html")))
    if files:
        return Path(files[-1]).read_text(encoding="utf-8", errors="ignore"), "saved copy"
    return None, "none"

def collect_parts_lines():
    lines = {c: {} for c in CANDIDATES}
    status = {}
    for sid in PARTS_SOURCES_USED:
        html, how = page_html(sid)
        if html is None:
            status[sid] = "not collected (offline and no saved copy)"
            continue
        soup = BeautifulSoup(html, "lxml")
        text = " ".join(soup.get_text(" ").split())
        if len(text) < 3000 and re.search(r"not a robot|verification required|captcha", text, re.I):
            status[sid] = "blocked by a bot check - not scraped"
            continue
        if sid == "S03":
            items = [a.get_text(" ", strip=True) for a in soup.select("a") if "Part No:" in a.get_text(" ", strip=True)]
            for it in items:
                for c in CANDIDATES:
                    if mentions(c, it):
                        lines[c][sid] = lines[c].get(sid, 0) + 1
            n = len(items)
        elif sid == "S04":
            try:
                n = len(max(pd.read_html(StringIO(html), header=None), key=len))
            except ValueError:
                n = 0
        elif sid == "S06":
            n = len(soup.select('a[href*="/part/"]'))
        else:
            status[sid] = f"page opened ({how}) but this notebook has no parser for it"
            continue
        if sid in PAGE_SCOPE and n:
            lines[PAGE_SCOPE[sid]][sid] = n
        status[sid] = f"read ({how}): {n} catalogue lines"
    return lines, status

PARTS_LINES, PARTS_STATUS = collect_parts_lines()

def parts_status_text(c):
    out = [f"{sid}: {n} lines" for sid, n in PARTS_LINES[c].items()]
    out += [f"{sid}: {PARTS_STATUS[sid]}" for sid, scope in PAGE_SCOPE.items() if scope == c and sid not in PARTS_LINES[c]]
    return "; ".join(out) if out else "no source in the pack lists this car"

for sid in PARTS_SOURCES_USED:
    print("Parts source", sid, "-", PARTS_STATUS.get(sid, "not collected"))


## 3. Collect

In [ ]:
# ---- 1. Adey Market ----
adey_rows, adey_method, adey_notes = collect_adey_cards()
adey = pd.DataFrame(adey_rows).drop_duplicates("listing_url") if adey_rows else pd.DataFrame(
    columns=["page_number", "listing_url", "price", "card_text", "collection_date"])
if len(adey):
    adey["vehicle"] = adey["card_text"].apply(match_candidate)
    adey["price_etb"] = adey["price"].apply(parse_price_etb)
    adey["source_id"], adey["collection_method"] = "S01", adey_method
    adey.to_csv(RAW / "adey_listings_scraped.csv", index=False)
    print(f"Adey: {len(adey)} unique listings ({adey_method})")
    print("Most common unmatched listings (check that none of your five is hiding here):")
    print(adey.loc[adey.vehicle == "Other", "card_text"].str.slice(0, 70).value_counts().head(8).to_string())
for n in adey_notes:
    print("NOTE:", n)

# ---- 2. Gezat (allowed pages only) ----
gezat, gezat_notes = collect_gezat()
gezat.to_csv(RAW / "gezat_scraped.csv", index=False)
for n in gezat_notes:
    print("NOTE:", n)
print("Gezat evidence found in the allowed pages:", int(gezat["gezat_listed"].notna().sum()), "of 5 candidates")

Adey: 240 unique listings (scraped (live, plain HTML))
Most common unmatched listings (check that none of your five is hiding here):
card_text
Telegram | New | The 2026/07 BYD e7 520 range variant is a mid-size pu    2
Telegram | New | White And Black Available | 6,700,000 | ETB | BeshBes    2
Telegram | 3,500,000 Bank loan will be facilitated | 6,500,000 | ETB |    2
Telegram | New | BYD Yuan Plus 2024/12 | 5,400,000 | ETB | Abena mekin    1
Telegram | New | BYD Yuan Plus 2024/12 | 5,400,000 | ETB | Bisre cars     1
Telegram | New | WULING EV CAR (በባንክ አያይዘን እንሸጣለን) | 6,450,000 | ETB |    1
Telegram | New | BYD Sealion 06 Ev | 8,200,000 | ETB | አሚር Amir የመኪና ሽ    1
Telegram | New | 2025 BYD Song Plus EV | 7,400,000 | ETB | Brand New C    1
NOTE: page limit (10) reached - counts may be incomplete
NOTE: homepage text length: 3614 characters
NOTE: sitemap addresses read as text: 22 (none of them opened)
Gezat evidence found in the allowed pages: 5 of 5 candidates


## 4. Count, rank and test the ranking

In [ ]:
rows = []
for i, c in enumerate(CANDIDATES):
    sub = adey[adey["vehicle"] == c] if len(adey) else adey
    man = manual.loc[c]
    if man["adey_manual_count"] != "":
        count, src, day = float(man["adey_manual_count"]), "manual", man["adey_manual_date"]
    elif len(adey):
        count, src, day = float(len(sub)), adey_method, ";".join(sorted(set(adey["collection_date"]))[:2])
    else:
        count, src, day = np.nan, "unknown", ""
    g = gezat.set_index("vehicle").loc[c]
    if man["gezat_listed"] in ("0", "1"):
        g_listed, g_src, g_ev = float(man["gezat_listed"]), "manual", man["gezat_note"] or man["gezat_evidence_url"]
    else:
        g_listed, g_src, g_ev = g["gezat_listed"], "scraped (allowed pages)", g["gezat_evidence"]
        if np.isnan(g_listed):
            g_src, g_ev = "unknown", "not found in the allowed pages - check the catalogue by hand"
    psrc = [sid for sid, n in PARTS_LINES[c].items() if n > 0]
    n_parts = len(psrc)
    rows.append({"vehicle_id": f"V{i+1:02d}", "vehicle": c,
                 "adey_listing_count": count, "adey_source": src, "adey_collection_date": day,
                 "adey_n_priced": int(sub["price_etb"].notna().sum()) if len(sub) else 0,
                 "adey_price_min_etb": sub["price_etb"].min() if len(sub) else np.nan,
                 "adey_price_median_etb": sub["price_etb"].median() if len(sub) else np.nan,
                 "adey_price_max_etb": sub["price_etb"].max() if len(sub) else np.nan,
                 "gezat_listed": g_listed, "gezat_source": g_src, "gezat_evidence": g_ev,
                 "parts_sources_found": ";".join(psrc), "n_parts_sources": n_parts,
                 "parts_lines_found": int(sum(PARTS_LINES[c].values())), "parts_status": parts_status_text(c)})
sel = pd.DataFrame(rows)

top = sel["adey_listing_count"].max()
sel["adey_share_of_max"] = sel["adey_listing_count"] / top if (top and top > 0) else np.nan
sel["parts_evidence_scaled"] = sel["n_parts_sources"].clip(upper=2) / 2
comp = pd.DataFrame({"adey_listings": sel["adey_share_of_max"], "gezat_listed": sel["gezat_listed"],
                     "parts_evidence": sel["parts_evidence_scaled"]})

def sel_score(c, w):
    return sum(w[k] * c[k].fillna(0) for k in w)

sel["selection_score"] = sel_score(comp, WEIGHTS_SEL).round(3)
sel = sel.sort_values(["selection_score", "adey_listing_count"], ascending=False, na_position="last").reset_index(drop=True)
sel["rank"] = range(1, len(sel) + 1)
if sel["selection_score"].duplicated().any():
    print("NOTE: some candidates have equal scores; ties are broken by the number of Adey listings.")

# does the ranking survive different weights?
scen = {"Base": WEIGHTS_SEL,
        "Adey listings only": {"adey_listings": 1.0, "gezat_listed": 0.0, "parts_evidence": 0.0},
        "Visibility only (Adey + Gezat)": {"adey_listings": 0.7, "gezat_listed": 0.3, "parts_evidence": 0.0},
        "Equal weights": {k: 1 / len(ACTIVE_SEL) for k in ACTIVE_SEL},
        "Parts-heavy (what-if)": {"adey_listings": 0.3, "gezat_listed": 0.2, "parts_evidence": 0.5}}
base_idx = pd.DataFrame(rows)["vehicle"]
comp.index = base_idx
ranks = pd.DataFrame({name: sel_score(comp, w).rank(ascending=False, method="min").astype(int) for name, w in scen.items()})
ranks.to_csv(PROC / "vehicle_selection_sensitivity.csv")
rng = np.random.default_rng(42)
hits = pd.Series(0.0, index=comp.index)
for w in rng.dirichlet(np.ones(len(ACTIVE_SEL)), size=1000):
    s = sel_score(comp, dict(zip(ACTIVE_SEL, w)))
    hits[s.nlargest(3).index] += 1
sel["pct_runs_in_top3"] = sel["vehicle"].map((hits / 10).round(1))
sel["suggested_top3"] = np.where(sel["rank"] <= 3, "Y", "N")
sel["selected"] = np.where(sel["vehicle"].isin(SELECTED), "Y", "N")
sel["target_market_relevance"] = sel.apply(lambda r: (
    f"{'unknown' if np.isnan(r['adey_listing_count']) else int(r['adey_listing_count'])} Adey Market listings (visibility proxy, not sales); "
    f"Gezat: {r['gezat_evidence'] or 'unknown'}; part-data sources: {r['parts_sources_found'] or 'none found'}"), axis=1)
sel.to_csv(PROC / "vehicle_selection.csv", index=False)

show = ["rank", "vehicle", "adey_listing_count", "gezat_listed", "n_parts_sources", "parts_lines_found", "selection_score", "pct_runs_in_top3", "suggested_top3", "selected"]
print("\nNOTE: parts are collected for all five cars (n_parts_sources, parts_lines_found, parts_status) but are not in the selection score, because each source in the pack is about one car or brand.")
print("\nCandidate ranking (listing counts are a visibility proxy, NOT registrations or sales):")
print(sel[show].to_string(index=False))
print("\nRank under other weights (1 = best):")
print(ranks.loc[sel["vehicle"]].to_string())

if sel["adey_listing_count"].isna().all() and sel["gezat_listed"].isna().all():
    print("\nWARNING: no Adey or Gezat evidence was obtained, so this ranking is NOT meaningful.")
    print("         Fill vehicle_manual_inputs.csv by hand (see data/raw/README.md) and run this step again.")
else:
    print("\nAll five candidates are analysed in notebooks 00-02 (50 vehicle-part combinations). The ranking above shows how visible each car is; it is not a cut-off.")



Candidate ranking (listing counts are a visibility proxy, NOT registrations or sales):
 rank               vehicle  adey_listing_count  gezat_listed  n_parts_sources  selection_score  pct_runs_in_top3 suggested_top3 selected
    1  Volkswagen ID.4/ID.6                26.0             1                1            0.900             100.0              Y        Y
    2           BYD Seagull                15.0             1                2            0.788             100.0              Y        Y
    3           BYD Yuan Up                20.0             1                0            0.685              49.2              Y        Y
    4           Toyota bZ4X                 7.0             1                1            0.535              50.8              N        N
    5 Changan Benben E-Star                 1.0             1                0            0.319               0.0              N        N

Rank under other weights (1 = best):
                       Base  Adey listings onl

## 5. Figure and source log

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
order = sel["vehicle"].tolist()[::-1]
chosen = ["#1f6f5c" if v in SELECTED else "#b8c4c2" for v in order]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
counts = sel.set_index("vehicle").loc[order, "adey_listing_count"].fillna(0)
axes[0].barh(order, counts, color=chosen)
axes[0].set_title("Adey Market listings (visibility proxy, not sales)")
for i, v in enumerate(counts):
    axes[0].text(v + 0.2, i, f"{int(v)}", va="center")
parts = pd.DataFrame({k: sel.set_index("vehicle").loc[order, col].fillna(0) * WEIGHTS_SEL[k]
                      for k, col in [("adey_listings", "adey_share_of_max"), ("gezat_listed", "gezat_listed"),
                                     ("parts_evidence", "parts_evidence_scaled")]})
left = np.zeros(len(parts))
for k, colr in zip(parts.columns, ["#1f6f5c", "#e0a030", "#3b6ea5"]):
    axes[1].barh(parts.index, parts[k], left=left, label=k.replace("_", " "), color=colr)
    left += parts[k].values
axes[1].set_title("Selection score (dark green on the left = your selected three)")
axes[1].legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout(); fig.savefig(FIG / "00_vehicle_selection.png", dpi=200); plt.close(fig)
write_source_log()
print("Saved figure/00_vehicle_selection.png and updated source_log.csv")

Saved figure/00_vehicle_selection.png and updated source_log.csv
